# Phase 11 — Vérification du parcours Predict–Optimize–Adapt

Les observations de trafic, les événements, les commandes et les coûts sont synthétiques. Le cas de 12 clients emploie un sous-ensemble de la matrice OSRM archivée de Casablanca et le modèle linéaire de phase 5. Le solveur utilise des durées robustes, puis le validateur rejoue le plan à capacité physique et à capacité réservée. B4 conserve ce plan et B5 choisit les replans aux événements éligibles.

Avec 10 caisses de capacité, une charge de départ de 9, une livraison de 3 et une collecte de 5 donnent 11 caisses à bord. C'est une violation malgré un départ admissible. La perte appariée se calcule par $L_{B4} - L_{B5}$. Elle peut diminuer quand le coût d'exploitation augmente si l'amélioration de service compense ce coût.

In [1]:
from math import isclose
from pathlib import Path
from resource import RUSAGE_SELF, getrusage
from time import perf_counter
import os
import sys

import pandas as pd

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "PROJECT_SPEC.md").is_file())
os.chdir(root)
sys.path.insert(0, str(root))

from dashboard import engine

started = perf_counter()
case = engine.prepare(engine.Inputs())
plan = engine.plan(case, "Robust ML")
fixed = engine.replay(case, plan, "B4")
adaptive = engine.replay(case, plan, "B5")
assert len(plan.report.unserved) == 0
assert fixed.metrics["replan_attempts"] == 0
assert all(isclose(run.frames[-1].loss_mad, run.metrics["objective_mad"])
           for run in (fixed, adaptive))
assert fixed.frames[-1].events == adaptive.frames[-1].events

print(f"Case {case.case_id[:12]} | {case.inputs.customers} customers | "
      f"{len(case.problem.vehicles)} available vehicles | seed {case.inputs.seed}")
print(f"Preparation {case.wall_s:.3f} s | validated planning {plan.wall_s:.3f} s")
print(f"Paired workflow {perf_counter() - started:.3f} s | "
      f"peak RSS {getrusage(RUSAGE_SELF).ru_maxrss / 1024:.1f} MiB")
print(f"Paired loss savings: {fixed.metrics['objective_mad'] - adaptive.metrics['objective_mad']:+.2f} MAD")
pd.DataFrame([{"Policy": run.policy, "Operational loss (MAD)": run.metrics["objective_mad"],
               "Operating cost (MAD)": run.metrics["operating_cost_mad"], "OTIF (%)": run.metrics["otif"] * 100,
               "Urgent orders served": run.metrics["urgent_served"],
               "Attempts": run.metrics["replan_attempts"], "Applied replans": run.metrics["replans_applied"],
               "Replay time (s)": run.wall_s} for run in (fixed, adaptive)]).round(3)

Case e3e4f3483929 | 12 customers | 3 available vehicles | seed 317
Preparation 0.895 s | validated planning 0.654 s
Paired workflow 1.735 s | peak RSS 486.5 MiB
Paired loss savings: +4.73 MAD


,Policy,Operational loss (MAD),Operating cost (MAD),OTIF (%),Urgent orders served,Attempts,Applied replans,Replay time (s)
0,B4,646.424,555.564,84.615,0,0,0,0.044
1,B5,641.694,556.194,92.308,0,3,1,0.141


La comparaison est calculée sur une seule journée avec le plan, les commandes, le trafic et les événements communs. Les résultats illustrent le comportement de l'interface. Les conclusions et l'incertitude multi-scénarios suivent les protocoles archivés des [phases 10](../docs/final_benchmark.md) et [10.1](../docs/scalability.md).